# 04 - Feature engineering

Construcción de retardos (`lag`), medias móviles y variables temporales sobre la capa Gold, respetando en todo momento el orden temporal para evitar data leakage. Al final se define el split cronológico de entrenamiento, validación y prueba.

## 1. Carga de la capa Gold

In [1]:
import pandas as pd
import numpy as np

gold = pd.read_parquet("../data/gold/gold_diario_2021_2025.parquet")

gold = gold.sort_values("fecha").reset_index(drop=True)

print(gold.shape)
gold.head()

(1825, 16)


,generacion_fotovoltaica,porcentaje_fotovoltaica,fecha,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol,dia_semana,mes,dia_año
0,34247.163,0.021989,2021-01-02,3.371410,-0.635081,7.377859,0.243681,3.248500,10.651848,66.304925,961.451484,957.727905,6.313311,5,1,2
1,37134.485,0.024855,2021-01-03,3.410448,-1.765693,8.584577,0.232906,2.664880,9.554461,64.446681,962.750448,959.676937,7.293419,6,1,3
2,26307.981,0.017347,2021-01-04,3.357818,-1.088909,7.805587,0.434375,2.175468,8.057505,73.948426,959.948097,956.382750,4.613515,0,1,4
3,29746.352,0.020739,2021-01-05,2.951290,-1.842792,7.745667,0.328913,1.645162,6.131043,74.186674,958.377786,955.568340,5.315905,1,1,5
4,25273.930,0.018880,2021-01-06,2.799222,-2.215370,7.812312,3.824368,1.976831,7.214517,71.192401,959.602209,955.569734,4.533651,2,1,6


In [ ]:

assert gold["fecha"].duplicated().sum() == 0, "Hay fechas duplicadas"
assert gold["fecha"].is_monotonic_increasing, "Las fechas no estan ordenadas"

rango_completo = pd.date_range(gold["fecha"].min(), gold["fecha"].max(), freq="D")
huecos = rango_completo.difference(gold["fecha"])
print("Dias faltantes en la capa Gold:", len(huecos))
if len(huecos) > 0:
    print(huecos)

Dias faltantes en la capa Gold: 0


## 2. Retardos (lags) de la generación fotovoltaica

`shift()` sobre la serie ya ordenada por fecha: cada fila solo puede ver generación de días **anteriores**, nunca del día que se pretende predecir ni de días posteriores.

In [3]:
gold["generacion_lag_1"] = gold["generacion_fotovoltaica"].shift(1)
gold["generacion_lag_7"] = gold["generacion_fotovoltaica"].shift(7)

gold[["fecha", "generacion_fotovoltaica", "generacion_lag_1", "generacion_lag_7"]].head(10)

,fecha,generacion_fotovoltaica,generacion_lag_1,generacion_lag_7
0,2021-01-02,34247.163,NaN,NaN
1,2021-01-03,37134.485,34247.163,NaN
2,2021-01-04,26307.981,37134.485,NaN
3,2021-01-05,29746.352,26307.981,NaN
4,2021-01-06,25273.930,29746.352,NaN
5,2021-01-07,9459.079,25273.930,NaN
6,2021-01-08,6751.867,9459.079,NaN
7,2021-01-09,13444.272,6751.867,34247.163
8,2021-01-10,18993.030,13444.272,37134.485
9,2021-01-11,33767.335,18993.030,26307.981


## 3. Media móvil de generación

In [ ]:

gold["generacion_media_7d"] = (
    gold["generacion_fotovoltaica"].shift(1).rolling(window=7, min_periods=7).mean()
)

gold[["fecha", "generacion_fotovoltaica", "generacion_media_7d"]].head(10)

,fecha,generacion_fotovoltaica,generacion_media_7d
0,2021-01-02,34247.163,NaN
1,2021-01-03,37134.485,NaN
2,2021-01-04,26307.981,NaN
3,2021-01-05,29746.352,NaN
4,2021-01-06,25273.930,NaN
5,2021-01-07,9459.079,NaN
6,2021-01-08,6751.867,NaN
7,2021-01-09,13444.272,24131.551000
8,2021-01-10,18993.030,21159.709429
9,2021-01-11,33767.335,18568.073000


## 4. Control de leakage — verificación explícita

Antes de seguir, se comprueba de forma programática que ninguna de las variables retardadas usa información del propio día o de días futuros.

In [ ]:

comparacion = gold["generacion_lag_1"].iloc[1:].reset_index(drop=True) == \
              gold["generacion_fotovoltaica"].iloc[:-1].reset_index(drop=True)

print("Coincidencias lag_1 con el dia anterior real:", comparacion.sum(), "/", len(comparacion))


primera_fila_valida = gold["generacion_media_7d"].first_valid_index()
print("Primera fila con media movil valida (indice):", primera_fila_valida, "-> deberia ser 7 (0-indexado)")

Coincidencias lag_1 con el dia anterior real: 1824 / 1824
Primera fila con media movil valida (indice): 7 -> deberia ser 7 (0-indexado)


## 5. Nota pendiente sobre la meteorología como feature

In [ ]:

variables_meteo = [
    "tmed", "tmin", "tmax", "prec",
    "velmedia", "racha",
    "hrMedia", "presMax", "presMin",
    "sol"
]

for var in variables_meteo:
    gold[f"{var}_lag_1"] = gold[var].shift(1)

gold[["fecha"] + [f"{v}_lag_1" for v in variables_meteo]].head()

,fecha,tmed_lag_1,tmin_lag_1,tmax_lag_1,prec_lag_1,velmedia_lag_1,racha_lag_1,hrMedia_lag_1,presMax_lag_1,presMin_lag_1,sol_lag_1
0,2021-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-03,3.371410,-0.635081,7.377859,0.243681,3.248500,10.651848,66.304925,961.451484,957.727905,6.313311
2,2021-01-04,3.410448,-1.765693,8.584577,0.232906,2.664880,9.554461,64.446681,962.750448,959.676937,7.293419
3,2021-01-05,3.357818,-1.088909,7.805587,0.434375,2.175468,8.057505,73.948426,959.948097,956.382750,4.613515
4,2021-01-06,2.951290,-1.842792,7.745667,0.328913,1.645162,6.131043,74.186674,958.377786,955.568340,5.315905


## 6. Eliminación de filas iniciales sin histórico suficiente

Las primeras filas no tienen `lag_7` ni `generacion_media_7d` disponibles (no hay 7 días previos). Se descartan, ya que no se pueden usar para entrenar ni evaluar de forma realista.

In [7]:
filas_antes = gold.shape[0]

gold_modelado = gold.dropna(
    subset=["generacion_lag_1", "generacion_lag_7", "generacion_media_7d"]
).reset_index(drop=True)

print("Filas antes:", filas_antes)
print("Filas despues de eliminar el arranque sin historico:", gold_modelado.shape[0])
print("Primera fecha valida para modelado:", gold_modelado["fecha"].min())

Filas antes: 1825
Filas despues de eliminar el arranque sin historico: 1818
Primera fecha valida para modelado: 2021-01-09 00:00:00


## 7. Split temporal (entrenamiento / validación / prueba)

División cronológica, sin barajar filas. El conjunto de prueba queda aislado y solo se usará en la evaluación final del modelo seleccionado.

In [8]:
fecha_min = gold_modelado["fecha"].min()
fecha_max = gold_modelado["fecha"].max()
print("Rango disponible para modelado:", fecha_min, "-", fecha_max)

# Ajusta estos cortes segun el reparto que quieras (ejemplo: ~70/15/15)
fin_train = "2023-12-31"
fin_val = "2024-12-31"

train = gold_modelado[gold_modelado["fecha"] <= fin_train]
val = gold_modelado[(gold_modelado["fecha"] > fin_train) & (gold_modelado["fecha"] <= fin_val)]
test = gold_modelado[gold_modelado["fecha"] > fin_val]

print("Train:", train.shape[0], train["fecha"].min(), "-", train["fecha"].max())
print("Val:  ", val.shape[0], val["fecha"].min(), "-", val["fecha"].max())
print("Test: ", test.shape[0], test["fecha"].min(), "-", test["fecha"].max())

assert train["fecha"].max() < val["fecha"].min(), "Train y val se solapan"
assert val["fecha"].max() < test["fecha"].min(), "Val y test se solapan"
print("\nOK: los tres conjuntos son cronologicamente disjuntos")

Rango disponible para modelado: 2021-01-09 00:00:00 - 2025-12-31 00:00:00
Train: 1087 2021-01-09 00:00:00 - 2023-12-31 00:00:00
Val:   366 2024-01-01 00:00:00 - 2024-12-31 00:00:00
Test:  365 2025-01-01 00:00:00 - 2025-12-31 00:00:00

OK: los tres conjuntos son cronologicamente disjuntos


## 8. Guardado de salidas

In [9]:
gold_modelado.to_parquet(
    "../data/gold/gold_features_2021_2025.parquet",
    index=False
)

train.to_parquet("../data/gold/train.parquet", index=False)
val.to_parquet("../data/gold/val.parquet", index=False)
test.to_parquet("../data/gold/test.parquet", index=False)

print("Guardado gold_modelado:", gold_modelado.shape)
print("Guardado train/val/test:", train.shape, val.shape, test.shape)

Guardado gold_modelado: (1818, 29)
Guardado train/val/test: (1087, 29) (366, 29) (365, 29)
